In [3]:
# scripts/make_dashboard.py
"""
Static Plotly dashboard (no server)

What this script does (for new contributors):
- Loads the pending-aware clean CSV from scripts/make_eda.py
- Validates key assumptions BEFORE plotting; if errors are found, writes
  outputs/validation_report.csv and aborts to avoid misleading visuals
- Builds the visuals you asked for:

Core visuals
  1) Big treemap (ITC vs Non-ITC → country) with word-wrapped country labels
  2) WG donut (participants per WG; overlaps allowed)
  3) WG participants stacked by ITC vs Non-ITC
  4) WG × WG co-membership heatmap
  5) 3-set Venn (ITC ∩ Any-WG ∩ MC) — STRICT: excludes rows with Pending WG/MC
     and shows how many rows were excluded due to Pending/Unknown

Extras (kept)
  6) Top countries bar (Total participants)
  7) Status overview (Yes/No/Pending) for MC/Core/WGs
  8) MC × Core overlap bar

(You asked to DROP these two visuals, so they are removed):
  ✗ Top countries by Any-WG rate
  ✗ Distribution of “# WGs per person”

Console/CSV debug outputs
- Prints and exports the triple-intersection rows (ITC ∧ Any-WG ∧ MC)
- Prints and exports the 2-set intersection rows (ITC ∧ MC) regardless of WG
- Prints strict “ITC only” and “MC only” counts (where WG & the other flag are KNOWN False)
"""

from __future__ import annotations
from pathlib import Path
import re, sys, shutil
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# ---------- repo paths ----------
def _find_repo_root() -> Path:
    """Make script robust when run from repo root, scripts/, or notebooks."""
    try:
        here = Path(__file__).resolve()
        return here.parent.parent
    except NameError:
        cwd = Path.cwd().resolve()
        if (cwd / "data").is_dir() and (cwd / "scripts").is_dir(): return cwd
        if cwd.name == "scripts" and (cwd.parent / "data").is_dir(): return cwd.parent
        cur = cwd
        for _ in range(5):
            if (cur / ".git").is_dir() or ((cur / "data").is_dir() and (cur / "scripts").is_dir()): return cur
            cur = cur.parent
        return cwd

ROOT      = _find_repo_root()
CSV_PATH  = ROOT / "data" / "processed" / "data_clean.csv"
DOCS_DIR  = ROOT / "docs"
OUT_DIR   = ROOT / "outputs"
DL_DIR    = DOCS_DIR / "downloads"
DOCS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DL_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root →", ROOT)
if not CSV_PATH.exists():
    sys.exit(f"ERROR: {CSV_PATH.relative_to(ROOT)} not found. Run scripts/make_eda.py first.")

# ---------- theme ----------
pio.templates.default = "plotly_white"
px.defaults.template  = "plotly_white"
px.defaults.height    = 420

ACCENT       = "#1F5FCC"   # blue
ITC_GREEN    = "#147D64"   # green (accessible)
NONITC_GREY  = "#6B7280"   # neutral grey
MC_ORANGE    = "#F97316"   # orange for MC ring
HEAT_BLUE    = "Blues"

# ---------- helpers ----------
DIGIT_RE = re.compile(r"\d+")
def wg_pretty(name: str) -> str:
    """Map 'wg1' → 'WG 1' for nicer labels."""
    m = DIGIT_RE.search(str(name)); return f"WG {m.group(0)}" if m else str(name)

def tweak(fig, title=None, height=None):
    """Consistent layout polish for Plotly figures."""
    if title: fig.update_layout(title=title)
    if height: fig.update_layout(height=height)
    fig.update_layout(
        margin=dict(l=10, r=10, t=60, b=10),
        hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    return fig

def plotly_to_html(fig: go.Figure, include_js=False) -> str:
    """Convert a Plotly figure to an embeddable HTML <div> (no full HTML shell)."""
    return pio.to_html(fig, include_plotlyjs=include_js, full_html=False,
                       default_width="100%", default_height="100%")

def card(fig_html: str, extra_class: str = "") -> str:
    """Wrap a figure HTML snippet in a card container for the grid layout."""
    if not fig_html: return ""
    cls = f"card {extra_class}".strip()
    return f"<div class='{cls}'>{fig_html}</div>"

def as_nullable_bool(s: pd.Series) -> pd.Series:
    """
    Ensure pandas nullable boolean dtype (True/False/<NA>).
    If input is object with 'true'/'false' tokens, coerce safely.
    """
    if pd.api.types.is_bool_dtype(s) and str(s.dtype) == "boolean":
        return s
    try:
        return s.astype("boolean")
    except Exception:
        raw = s.astype(str).str.strip().str.lower()
        mapped = raw.map({
            "true": True, "t": True, "1": True, "yes": True, "y": True, "member": True, "x": True,
            "false": False, "f": False, "0": False, "no": False, "n": False
        })
        return mapped.astype("boolean")

def bool_for_counts(s: pd.Series) -> pd.Series:
    """
    For counting at aggregate level, treat <NA> as False
    (prevents overstating participation due to pending/unknown).
    Returns plain bool dtype for fast ops.
    """
    return as_nullable_bool(s).fillna(False).astype(bool)

# ---------- load & gentle clean ----------
df = pd.read_csv(CSV_PATH, low_memory=False)

# Normalize all object columns' whitespace (non-destructive)
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).str.replace("\u00A0", " ", regex=False).str.strip()

# Country column
country_col = "country_clean" if "country_clean" in df.columns else ("country" if "country" in df.columns else None)
if not country_col: sys.exit("ERROR: no 'country_clean' or 'country' column.")

# Flags (nullable booleans expected from EDA)
mc_col   = "mc_member"  if "mc_member"  in df.columns else None
core_col = "core_group" if "core_group" in df.columns else None

# --- Robust WG detection + de-dup by WG number (handles wg1, wg_1, etc.) ---
raw_wg_cols = [c for c in df.columns if re.match(r"(?i)^wg[_]*\d+", c)]
wg_buckets: dict[int, list[str]] = {}
for col in raw_wg_cols:
    m = re.search(r"\d+", col)
    if not m: continue
    num = int(m.group(0))
    wg_buckets.setdefault(num, []).append(col)

wg_cols: list[str] = []
for num in sorted(wg_buckets):
    cols = wg_buckets[num]
    canonical = cols[0]
    # OR-merge duplicates into canonical; NA treated as False for OR
    base = bool_for_counts(df.get(canonical, False))
    for other in cols[1:]:
        base = base | bool_for_counts(df[other])
    df[canonical] = pd.Series(base, dtype="boolean")
    wg_cols.append(canonical)

wg_member_col = "wg_member" if "wg_member" in df.columns else None

# Ensure MC/Core/any_wg are nullable booleans (preserve <NA>)
if mc_col:   df[mc_col]   = as_nullable_bool(df[mc_col])
else:        df["mc_member"] = pd.Series([False]*len(df), dtype="boolean"); mc_col = "mc_member"

if core_col: df[core_col] = as_nullable_bool(df[core_col])
else:        df["core_group"] = pd.Series([False]*len(df), dtype="boolean"); core_col = "core_group"

# any_wg: keep from EDA (nullable); if missing, derive from WGs
if "any_wg" not in df.columns:
    if wg_cols:
        df["any_wg"] = pd.Series(df[wg_cols].fillna(False).any(axis=1), dtype="boolean")
    else:
        df["any_wg"] = pd.Series([False]*len(df), dtype="boolean")
df["any_wg"] = as_nullable_bool(df["any_wg"])

# ITC → boolean for analysis (source is Yes/No string from EDA)
have_itc = "itc_countries" in df.columns
if have_itc:
    itc_norm = df["itc_countries"].astype(str).str.strip().str.lower()
    df["itc_bool"] = itc_norm.isin(["yes","y","true","1"])
else:
    df["itc_bool"] = False

# ---------- aggregates (country level) ----------
# Use lambda with NA→False so “Pending” isn’t over-counted in country totals
agg_parts = {
    "Country":      (country_col, "first"),
    "Total":        (country_col, "size"),
    "MC":           (mc_col,   lambda s: s.fillna(False).sum()),
    "Core":         (core_col, lambda s: s.fillna(False).sum()),
    "Any WG":       ("any_wg", lambda s: s.fillna(False).sum()),
}
if have_itc:
    agg_parts["ITC participants"] = ("itc_bool", "sum")

for c in wg_cols:
    agg_parts[c] = (c, lambda s: s.fillna(False).sum())

country_agg = df.groupby(country_col, as_index=False).agg(**agg_parts)

# ensure integer dtype for counts
for c in ["Total", "MC", "Core", "Any WG"] + (["ITC participants"] if have_itc else []) + wg_cols:
    if c in country_agg.columns:
        country_agg[c] = country_agg[c].fillna(0).astype(int)

country_agg["ITC group"] = np.where(country_agg.get("ITC participants", 0) > 0, "ITC", "Non-ITC")
country_agg["AnyWG_rate"] = np.where(country_agg["Total"] > 0, country_agg["Any WG"] / country_agg["Total"], 0.0)

# KPIs (kept simple and robust)
kpi_total_people  = int(df.shape[0])
kpi_countries     = int(country_agg["Country"].nunique())
kpi_any_wg_total  = int(country_agg["Any WG"].sum())
kpi_itc_countries = int((country_agg["ITC group"]=="ITC").sum()) if have_itc else 0

# WG summaries for plots (count True only; Pending/NA excluded)
if wg_cols:
    id_cols = [country_col, mc_col, core_col, "itc_bool"]
    melt = df[id_cols + wg_cols].melt(id_vars=id_cols, value_vars=wg_cols, var_name="WG", value_name="member")
    wg_long = melt[melt["member"] == True].copy()  # noqa: E712
    wg_summary = (wg_long.groupby("WG", as_index=False)
                  .agg(Participants=("member","size"),
                       Countries=(country_col,"nunique"),
                       ITC=("itc_bool","sum")))
    wg_summary["WG_label"] = wg_summary["WG"].map(wg_pretty)
    wg_order = wg_summary.sort_values("Participants", ascending=False)["WG_label"].tolist()
else:
    wg_long = pd.DataFrame(); wg_summary = pd.DataFrame(); wg_order = []

# ========================================================================== #
#                           DATA VALIDATION BLOCK                            #
# ========================================================================== #
STRICT = True  # if True, exit on ERRORs (warnings still allow plots)

issues: list[tuple[str, str]] = []  # (LEVEL, message)
def _add(level: str, msg: str):
    issues.append((level, msg)); print(f"{level}: {msg}")

# 1) Country column sanity
n_country_na = int(df[country_col].isna().sum())
if n_country_na > 0:
    _add("ERROR", f"{n_country_na} rows have missing '{country_col}'.")

# 2) Flag columns convertible to nullable boolean
flag_cols = [mc_col, core_col, "any_wg"] + wg_cols
for col in flag_cols:
    try:
        _ = as_nullable_bool(df[col])
    except Exception:
        _add("ERROR", f"Column '{col}' could not be coerced to nullable boolean.")

# 3) any_wg logical consistency (compare only where any_wg is not NA)
if wg_cols:
    calc_any = df[wg_cols].fillna(False).any(axis=1)
    mask = df["any_wg"].notna()
    mismatch = int((calc_any[mask].astype(bool) != df.loc[mask, "any_wg"].astype(bool)).sum())
    if mismatch:
        _add("ERROR", f"'any_wg' mismatches WG flags in {mismatch} rows (ignoring NA).")

# 4) Aggregate consistency
if int(country_agg["Total"].sum()) != len(df):
    _add("ERROR", f"Groupby total mismatch: sum(country_agg['Total'])={int(country_agg['Total'].sum())} "
                  f"!= len(df)={len(df)}.")
for c in ["Any WG", "MC", "Core"] + wg_cols + (["ITC participants"] if have_itc else []):
    if c in country_agg.columns:
        lhs = int(country_agg[c].sum())
        if c == "Any WG":
            rhs = int(bool_for_counts(df["any_wg"]).sum())
        elif c in wg_cols:
            rhs = int(bool_for_counts(df[c]).sum())
        elif c == "MC":
            rhs = int(bool_for_counts(df[mc_col]).sum())
        elif c == "Core":
            rhs = int(bool_for_counts(df[core_col]).sum())
        elif c == "ITC participants":
            rhs = int(df["itc_bool"].sum())
        else:
            rhs = lhs
        if lhs != rhs:
            _add("ERROR", f"Aggregate mismatch for '{c}': group sum={lhs} vs row sum={rhs}.")

# 5) ITC sanity (if present)
if have_itc:
    if (df["itc_bool"].dtype != bool) and (df["itc_bool"].dtype != np.bool_):
        _add("ERROR", f"itc_bool dtype is {df['itc_bool'].dtype}, expected bool.")
    vc = df["itc_bool"].value_counts(dropna=False)
    if int(vc.sum()) != len(df):
        _add("ERROR", "itc_bool value_counts does not sum to total rows.")

# 6) WG presence
if not wg_cols:
    _add("WARNING", "No WG columns detected; WG visuals will be skipped.")

# ---------- write report & maybe stop ----------
if issues:
    rep = pd.DataFrame(issues, columns=["level", "message"])
    out_rep = OUT_DIR / "validation_report.csv"
    rep.to_csv(out_rep, index=False, encoding="utf-8-sig")
    print(f"\nValidation report saved → {out_rep}")
    n_err = sum(1 for lvl, _ in issues if lvl == "ERROR")
    n_warn = sum(1 for lvl, _ in issues if lvl == "WARNING")
    print(f"Summary: {n_err} ERROR(s), {n_warn} WARNING(s).")
    if STRICT and n_err > 0:
        print("Aborting before visuals due to validation ERRORs. Fix data and rerun.")
        sys.exit(1)
else:
    print("Validation passed: no issues found.")

# ========================================================================== #
#                               VISUALS BELOW                                #
# ========================================================================== #

# A) Big Treemap (ITC vs Non-ITC → Country) with wrapped labels
country_agg["Country_wrapped"] = country_agg["Country"].str.replace(r"\s+", "<br>", regex=True)
fig_treemap_big = px.treemap(
    country_agg.sort_values("Total", ascending=False),
    path=["ITC group", "Country_wrapped"], values="Total",
    color="ITC group", color_discrete_map={"ITC": ITC_GREEN, "Non-ITC": NONITC_GREY},
    title="Treemap: Total people by country (grouped by ITC vs Non-ITC)"
)
fig_treemap_big.update_traces(
    textinfo="label+value",
    tiling=dict(pad=3),
    textfont=dict(size=14),
    hovertemplate="<b>%{label}</b><br>Total: %{value}<extra></extra>"
)
tweak(fig_treemap_big, height=880)

# B) WG donut — participants per WG
if not wg_summary.empty:
    pie_df = wg_summary.sort_values("Participants", ascending=False)
    fig_wg_donut = px.pie(
        pie_df,
        names="WG_label",
        values="Participants",
        hole=0.45,
        title="Working Groups — participants per WG",
    )

    # text inside slices
    fig_wg_donut.update_traces(
        textposition="inside",
        textinfo="percent+label"
    )

    # run your standard layout polish FIRST
    tweak(fig_wg_donut)

    # ...then override what tweak just set for the legend
    fig_wg_donut.update_layout(
        title_x=0.5,
        legend=dict(
            orientation="h",
            x=0.5,
            xanchor="center",
            y=-0.15,   # down below the donut
            yanchor="top",
        ),
        margin=dict(l=10, r=10, t=60, b=80),
    )
else:
    fig_wg_donut = None


# C) WG participants stacked by ITC vs Non-ITC
if not wg_long.empty:
    wg_itc_breakdown = (wg_long.assign(ITC=wg_long["itc_bool"].map({True:"ITC", False:"Non-ITC"}))
                                 .groupby(["WG","ITC"], as_index=False).size()
                                 .rename(columns={"size":"Count"}))
    wg_itc_breakdown["WG_label"] = wg_itc_breakdown["WG"].map(wg_pretty)
    if wg_order:
        wg_itc_breakdown["WG_label"] = pd.Categorical(wg_itc_breakdown["WG_label"], categories=wg_order, ordered=True)
        wg_itc_breakdown = wg_itc_breakdown.sort_values(["WG_label","ITC"])
    fig_wg_stack = px.bar(
        wg_itc_breakdown, x="WG_label", y="Count", color="ITC",
        barmode="stack", color_discrete_map={"ITC": ITC_GREEN, "Non-ITC": NONITC_GREY},
        title="Participants per WG (stacked by ITC vs Non-ITC)"
    )
    fig_wg_stack.update_xaxes(title="")
    tweak(fig_wg_stack)
else:
    fig_wg_stack = None

# D) WG × WG co-membership heatmap (NA treated as False)
if wg_cols:
    mat = df[wg_cols].fillna(False).astype(int)
    overlap = pd.DataFrame(mat.T.values @ mat.values, index=wg_cols, columns=wg_cols)
    np.fill_diagonal(overlap.values, 0)  # clarity
    overlap.index   = [wg_pretty(c) for c in overlap.index]
    overlap.columns = [wg_pretty(c) for c in overlap.columns]
    show_text = overlap.shape[0] <= 9
    fig_wg_overlap = go.Figure(data=go.Heatmap(
        z=overlap.values, x=list(overlap.columns), y=list(overlap.index),
        colorscale=HEAT_BLUE, zmin=0, zsmooth=False, colorbar=dict(title="Co-members")
    ))
    if show_text:
        fig_wg_overlap.add_trace(go.Scatter(
            x=np.repeat(list(overlap.columns), overlap.shape[0]),
            y=list(overlap.index) * overlap.shape[1],
            mode="text", text=[str(int(v)) for v in overlap.values.flatten()],
            textfont=dict(size=10), hoverinfo="skip", showlegend=False
        ))
    tweak(fig_wg_overlap, "WG × WG co-membership (counts)", height=520)
else:
    fig_wg_overlap = None

# E) STRICT 3-set Venn (ITC ∩ Any-WG ∩ MC)
def venn_itc_wg_mc_strict(df: pd.DataFrame) -> tuple[go.Figure, dict]:
    """
    STRICT Venn logic:
    - ITC uses plain boolean (True/False)
    - WG/MC must be KNOWN True/False. Rows where WG or MC is <NA> (Pending/Unknown)
      are excluded from the 7 regions; we show how many were excluded.
    Returns (figure, dict_of_masks_for_exports)
    """
    I  = df["itc_bool"].astype(bool)
    Wn = as_nullable_bool(df["any_wg"])
    Mn = as_nullable_bool(df["mc_member"])

    known_mask = Wn.notna() & Mn.notna()
    excl       = int((~known_mask).sum())

    # Masks (restrict to KNOWN rows)
    Ik = I[known_mask]
    Wt = Wn[known_mask].eq(True)
    Wf = Wn[known_mask].eq(False)
    Mt = Mn[known_mask].eq(True)
    Mf = Mn[known_mask].eq(False)

    # Region masks (index matches filtered df)
    m111 = ( Ik &  Wt &  Mt)     # ITC ∩ WG ∩ MC
    m110 = ( Ik &  Wt &  Mf)     # ITC ∩ WG (not MC)
    m101 = ( Ik &  Wf &  Mt)     # ITC ∩ MC (not WG)
    m011 = (~Ik &  Wt &  Mt)     # WG ∩ MC (not ITC)
    m100 = ( Ik &  Wf &  Mf)     # ITC only
    m010 = (~Ik &  Wt &  Mf)     # WG only
    m001 = (~Ik &  Wf &  Mt)     # MC only
    m000 = (~Ik &  Wf &  Mf)     # Neither (among known rows)

    n111 = int(m111.sum()); n110 = int(m110.sum()); n101 = int(m101.sum()); n011 = int(m011.sum())
    n100 = int(m100.sum()); n010 = int(m010.sum()); n001 = int(m001.sum()); n000 = int(m000.sum())
    total_known = n111+n110+n101+n011+n100+n010+n001+n000

    # Build figure
    fig = go.Figure()
    fig.update_xaxes(visible=False, range=[0, 10]); fig.update_yaxes(visible=False, range=[0, 10])

    r = 2.6
    # ITC
    fig.add_shape(type="circle", xref="x", yref="y",
                  x0=3.3-r, y0=4.0-r, x1=3.3+r, y1=4.0+r,
                  line=dict(color=ITC_GREEN, width=3), fillcolor="rgba(20,125,100,0.25)")
    # Any-WG
    fig.add_shape(type="circle", xref="x", yref="y",
                  x0=6.7-r, y0=4.0-r, x1=6.7+r, y1=4.0+r,
                  line=dict(color=ACCENT, width=3), fillcolor="rgba(31,95,204,0.25)")
    # MC
    fig.add_shape(type="circle", xref="x", yref="y",
                  x0=5.0-r, y0=6.8-r, x1=5.0+r, y1=6.8+r,
                  line=dict(color=MC_ORANGE, width=3), fillcolor="rgba(249,115,22,0.25)")

    # labels
    fig.add_annotation(x=2.1, y=7.3, text="<b>ITC</b>", showarrow=False, font=dict(size=14, color=ITC_GREEN))
    fig.add_annotation(x=7.9, y=7.3, text="<b>Any WG</b>", showarrow=False, font=dict(size=14, color=ACCENT))
    fig.add_annotation(x=5.0, y=9.3, text="<b>MC</b>", showarrow=False, font=dict(size=14, color=MC_ORANGE))

    # region counts (known-only)
    fig.add_annotation(x=2.1, y=4.0, text=f"{n100:,}", showarrow=False, font=dict(size=18))  # ITC only
    fig.add_annotation(x=7.9, y=4.0, text=f"{n010:,}", showarrow=False, font=dict(size=18))  # AnyWG only
    fig.add_annotation(x=5.0, y=8.8, text=f"{n001:,}", showarrow=False, font=dict(size=18))  # MC only
    fig.add_annotation(x=5.0, y=5.2, text=f"<b>{n111:,}</b>", showarrow=False, font=dict(size=20))  # all three
    fig.add_annotation(x=5.0, y=3.0, text=f"{n110:,}", showarrow=False, font=dict(size=16))  # ITC∩WG
    fig.add_annotation(x=4.0, y=6.3, text=f"{n101:,}", showarrow=False, font=dict(size=16))  # ITC∩MC
    fig.add_annotation(x=6.0, y=6.3, text=f"{n011:,}", showarrow=False, font=dict(size=16))  # WG∩MC

    fig.add_annotation(x=5.0, y=0.9,
                       text=f"Neither: {n000:,}  •  Unknown (Pending): {excl:,}  •  Total known: {total_known:,}",
                       showarrow=False, font=dict(size=12, color="#444"))

    fig.update_layout(title="Venn — ITC vs Any-WG vs MC (known only; Pending shown as Unknown)", showlegend=False)
    tweak(fig, height=520)

    # For exports: original index positions within KNOWN rows → map back to df index
    idx_known = df.index[known_mask]
    masks_for_export = {
        "ITC_and_WG_and_MC": idx_known[m111],
        "ITC_and_MC_not_WG": idx_known[m101],
        "ITC_and_WG_not_MC": idx_known[m110],
        "WG_and_MC_not_ITC": idx_known[m011],
        "ITC_only":          idx_known[m100],
        "WG_only":           idx_known[m010],
        "MC_only":           idx_known[m001],
        "Neither_known":     idx_known[m000],
        "Known_mask":        idx_known,
        "Excluded_pending":  df.index[~known_mask]
    }
    return fig, masks_for_export

fig_venn_itc_wg_mc, venn_masks = venn_itc_wg_mc_strict(df)

# ---------- EXTRA visuals (kept) ----------
TOPN = 15

# 6) Top countries by participants (bar)
bar_df = country_agg.sort_values("Total", ascending=False).head(TOPN)
fig_country_bar = px.bar(
    bar_df, y="Country", x="Total", orientation="h",
    title=f"Top {TOPN} countries by participants"
)
fig_country_bar.update_yaxes(categoryorder="total ascending", title="")
tweak(fig_country_bar)

# 7) Pending/Yes/No overview for MC/Core/WGs (uses *_status if present)
status_cols = []
if mc_col and f"{mc_col}_status" in df.columns:   status_cols.append(f"{mc_col}_status")
if core_col and f"{core_col}_status" in df.columns: status_cols.append(f"{core_col}_status")
for c in wg_cols:
    sc = f"{c}_status"
    if sc in df.columns:
        status_cols.append(sc)

if status_cols:
    st = (df[status_cols]
          .melt(var_name="flag", value_name="status")
          .dropna(subset=["status"]))
    st["flag"] = st["flag"].str.replace("_status$", "", regex=True)
    def nicify(s):
        if s.startswith("wg"): return s.upper()
        return s.replace("_", " ").title()
    st["flag_label"] = st["flag"].map(nicify)

    order_status = ["Yes", "No", "Pending"]
    st["status"] = pd.Categorical(st["status"], categories=order_status, ordered=True)
    stc = (st.groupby(["flag_label","status"], as_index=False)
             .size().rename(columns={"size":"Count"}))
    fig_pending = px.bar(
        stc, x="flag_label", y="Count", color="status",
        category_orders={"status": order_status},
        barmode="stack", title="Status by flag — Yes / No / Pending"
    )
    fig_pending.update_xaxes(title="")
    tweak(fig_pending)
else:
    fig_pending = None

# 8) MC × Core overlap (NA→False)
if mc_col and core_col:
    mc_b   = bool_for_counts(df[mc_col])
    core_b = bool_for_counts(df[core_col])
    cross = (pd.DataFrame({"MC": np.where(mc_b, "MC: Yes", "MC: No"),
                           "Core": np.where(core_b, "Core: Yes", "Core: No")})
             .value_counts().reset_index(name="Count"))
    fig_mc_core = px.bar(
        cross, x="MC", y="Count", color="Core", barmode="group",
        title="MC × Core overlap (NA→False)"
    )
    tweak(fig_mc_core)
else:
    fig_mc_core = None

# ---------- console debug + CSV exports ----------
# Strict "only" counts where WG is KNOWN False (not Pending)
Wn = as_nullable_bool(df["any_wg"])
Mn = as_nullable_bool(df["mc_member"])
I  = df["itc_bool"].astype(bool)
W_false_known = Wn.eq(False)
M_false_known = Mn.eq(False)

itc_only_strict = df.index[I & W_false_known & M_false_known]
mc_only_strict  = df.index[(~I) & W_false_known & Mn.eq(True)]

print("ITC only (strict, WG known False):", int(len(itc_only_strict)))
print("MC only  (strict, WG known False):", int(len(mc_only_strict)))

# "Middle": triple-intersection and 2-set ITC∩MC (any WG)
center_triple_idx = venn_masks["ITC_and_WG_and_MC"]
itc_mc_idx        = df.index[I & Mn.fillna(False)]  # regardless of WG (NA→False to be conservative)

cols_to_show = [country_col] + [c for c in ["name","fullname","email"] if c in df.columns]
print("Center (ITC ∧ Any-WG ∧ MC):")
print(df.loc[center_triple_idx, cols_to_show])
print("ITC ∧ MC (any WG):")
print(df.loc[itc_mc_idx, cols_to_show])

# Export CSVs for easy review
(pd.DataFrame(df.loc[center_triple_idx, cols_to_show])
   .to_csv(OUT_DIR / "venn_center_itc_wg_mc.csv", index=False, encoding="utf-8-sig"))
(pd.DataFrame(df.loc[itc_mc_idx, cols_to_show])
   .to_csv(OUT_DIR / "venn_itc_and_mc_anywg.csv", index=False, encoding="utf-8-sig"))
(pd.DataFrame(df.loc[itc_only_strict, cols_to_show])
   .to_csv(OUT_DIR / "venn_itc_only_strict.csv", index=False, encoding="utf-8-sig"))
(pd.DataFrame(df.loc[mc_only_strict, cols_to_show])
   .to_csv(OUT_DIR / "venn_mc_only_strict.csv", index=False, encoding="utf-8-sig"))

print("Saved →", (OUT_DIR / "venn_center_itc_wg_mc.csv").relative_to(ROOT))
print("Saved →", (OUT_DIR / "venn_itc_and_mc_anywg.csv").relative_to(ROOT))

# ---------- assemble HTML ----------
PLOTLY_CDN = '<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>'

kpi_html = f"""
<section class="kpis">
  <div class="kpi"><div class="kpi-num">{kpi_total_people:,}</div><div class="kpi-label">Total people</div></div>
  <div class="kpi"><div class="kpi-num">{kpi_countries:,}</div><div class="kpi-label">Countries</div></div>
  <div class="kpi"><div class="kpi-num">{kpi_any_wg_total:,}</div><div class="kpi-label">Any WG</div></div>
  {"<div class='kpi'><div class='kpi-num'>"+f"{kpi_itc_countries:,}"+"</div><div class='kpi-label'>ITC countries represented</div></div>" if have_itc else ""}
</section>
"""

# Prepare HTML snippets safely
first_chart_html = plotly_to_html(fig_treemap_big, include_js=True)
wg_donut_html    = plotly_to_html(fig_wg_donut) if fig_wg_donut else ""
wg_stack_html    = plotly_to_html(fig_wg_stack) if 'fig_wg_stack' in locals() and fig_wg_stack else ""
wg_overlap_html  = plotly_to_html(fig_wg_overlap) if fig_wg_overlap else ""
venn_html        = plotly_to_html(fig_venn_itc_wg_mc)

# kept extras
country_bar_html = plotly_to_html(fig_country_bar)
pending_html     = plotly_to_html(fig_pending) if fig_pending else ""
mc_core_html     = plotly_to_html(fig_mc_core) if fig_mc_core else ""

grid_html = f"""
<section class="grid">
  {card(first_chart_html, "span-all")}
  {card(wg_donut_html)}
  {card(wg_stack_html)}
  {card(wg_overlap_html, "span-2")}
  {card(venn_html, "span-2")}

  {card(country_bar_html, "span-2")}
  {card(pending_html)}
  {card(mc_core_html, "span-2")}
</section>
"""

page_html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <title>Agrifood Evidence — Dashboard</title>
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  {PLOTLY_CDN}
  <style>
    :root {{
      --accent:{ACCENT}; --text:#111827; --muted:#6B7280; --bg:#ffffff; --card:#F8FAFC; --line:#E5E7EB;
    }}
    * {{ box-sizing:border-box; }}
    html, body {{ margin:0; padding:0; background:var(--bg); color:var(--text);
                  font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif; }}
    .container {{ max-width: 1200px; margin: 1.25rem auto; padding: 0 1rem; }}
    header h1 {{ margin:0; font-size: 1.6rem; }}
    header .meta {{ color:var(--muted); margin:.25rem 0 1rem; }}

    .kpis {{ display:grid; grid-template-columns: repeat(auto-fit,minmax(160px,1fr)); gap:12px; margin: 1rem 0 1.25rem; }}
    .kpi {{ background:var(--card); border:1px solid var(--line); border-radius:12px; padding:14px; }}
    .kpi-num {{ font-size:1.8rem; font-weight:700; line-height:1; }}
    .kpi-label {{ color:var(--muted); font-size:.95rem; }}

    .grid {{ display:grid; gap:14px; grid-template-columns: repeat(auto-fit,minmax(320px,1fr)); }}
    .card {{ background:#fff; border:1px solid var(--line); border-radius:12px; padding:8px; min-height: 340px; }}
    .span-2 {{ grid-column: span 2; }}
    .span-all {{ grid-column: 1 / -1; }} /* take full row width */
  </style>
</head>
<body>
  <main class="container">
    <header>
      <h1>Agrifood Evidence — Dashboard</h1>
      <p class="meta">Interactive charts from <code>data/processed/data_clean.csv</code></p>
    </header>

    {kpi_html}
    {grid_html}
  </main>
</body>
</html>
"""

out_path = DOCS_DIR / "dashboard.html"
out_path.write_text(page_html, encoding="utf-8")
print("Saved →", out_path.relative_to(ROOT))

# (optional) export country_agg for download parity
out_country_csv = OUT_DIR / "dashboard_country_agg.csv"
country_agg.to_csv(out_country_csv, index=False, encoding="utf-8-sig")
shutil.copyfile(out_country_csv, DL_DIR / "country_agg.csv")
print("Saved →", (DL_DIR / "country_agg.csv").relative_to(ROOT))


Repo root → C:\Users\James\Documents\GitHub\evidence-map-agrifood
Validation passed: no issues found.
ITC only (strict, WG known False): 0
MC only  (strict, WG known False): 1
Center (ITC ∧ Any-WG ∧ MC):
              country_clean
0                   Türkiye
2                 Lithuania
8                   Croatia
10                   Cyprus
15                 Portugal
17                  Hungary
23                  Croatia
27                  Türkiye
32                   Poland
34               Montenegro
35                   Greece
36          North Macedonia
41                 Slovakia
47               Montenegro
50                   Greece
54                  Czechia
55   Bosnia and Herzegovina
57   Bosnia and Herzegovina
60                  Albania
63                  Estonia
70                Lithuania
89                   Serbia
110                 Albania
116                  Serbia
138                  Poland
ITC ∧ MC (any WG):
              country_clean
0                   T

C:\Users\James\AppData\Local\Temp\ipykernel_31772\1374073696.py:545: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

